# Modelado Predictivo: Clasificación con Regresión Logística (One-vs-Rest)
**Proyecto:** Clasificador de Salud Fetal (Dataset Cardiotocography - UCI).

**Contexto:** Tomando como base el dataset limpio, libre de fugas de datos ("X_features.csv" y "y_target.csv"), este notebook implementa y entrena un modelo de **Regresión Logística multiclase** mediante la estrategia **One-vs-Rest (OvR)** utilizando código escrito completamente desde cero (*scratch*) con ayuda del curso de Machine Learning. El propósito es evaluar la capacidad lineal de separación de las 21 variables para diagnosticar el estado fetal (`NSP`) en sus tres categorías (Normal, Sospechoso y Patológico).

### Objetivos del Modelado

1. **Construcción y Entrenamiento OvR:** Adaptar nuestro clasificador binario de regresión logística a un problema de 3 clases entrenando tres modelos especializados de forma independiente (Cada clase frente al resto).

2. **Decisión por Probabilidad (Argmax):** Combinar las probabilidades predichas por los tres modelos para asignar el diagnóstico final mediante la función de votación por probabilidad máxima (`np.argmax`).

3. **Evaluación Clínica y Curva ROC Multiclase:** Analizar el rendimiento mediante la Matriz de Confusión, Recall, F1-Score y la Curva ROC (con cálculo de AUC desde cero), comparando además el rendimiento frente a la implementación comercial de `scikit-learn`.

### Ficha Metodológica del Modelo (Sujeto a eliminacion / preguntar a dylan)

* **Descripción breve del método:** Es un clasificador lineal que transforma una combinación lineal de las características mediante la función sigmoide para estimar probabilidades. Para manejar 3 clases, se aplica la estrategia *One-vs-Rest*, entrenando 3 modelos independientes (`learning_rate=0.1`, `n_iterations=2000`).

* **Justificación de su uso:** Permite establecer una línea base lineal e interpretable de cómo operan los coeficientes y pesos de las 21 variables sobre las clases fetales, validando la robustez de nuestra optimización por descenso de gradiente programada a mano.

* **Métricas de evaluación utilizadas:** Exactitud Global (*Accuracy*), Matriz de Confusión, *Recall*, *Precisión*, *F1-Score* por clase, *Promedio Macro F1-Score* y el cálculo del *Área Bajo la Curva ROC (AUC)*.

* **Análisis de resultados:** Detallado al final del notebook, logrando un **88.97% de Exactitud Global** y un excelente desempeño en la detección de clases normales y patológicas mediante votación multiclase.

In [7]:
# Importación de librerías esenciales
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt


# Modificación para importar los modelos fuera de la carpeta actual
import os
import sys

root_dir = os.path.abspath(os.path.join(os.getcwd(), ".."))
if root_dir not in sys.path:
    sys.path.append(root_dir)

# Importación del modelo
from src.LogisticRegression import LogisticRegression
from src.utils_and_metrics import train_test_split, accuracy, confusion_matrix, recall, f1_score, precision, roc

#### **1. Preparación de Datos: Estandarización y Binarización One-vs-Rest**
A diferencia de la exclusión binaria, aquí utilizaremos todo el conjunto de datos (incluyendo la Clase 2). Para adaptar nuestro modelo binario a un problema de 3 clases, crearemos **3 vectores objetivo independientes**:
* **y_train_c1:** Positivo (1) si es Normal, Negativo (0) para el resto.
* **y_train_c2:** Positivo (1) si es Sospechoso, Negativo (0) para el resto.
* **y_train_c3:** Positivo (1) si es Patológico, Negativo (0) para el resto.

In [ ]:
#Cargar el dataset
df = pd.read_csv('../data/processed/CTG_clean.csv')

#Separacion de caracteristicas y variable objetivo
X_todo = df.drop(columns=['NSP']).to_numpy()
y_todo_og = df['NSP'].to_numpy().astype(int)

#Particion de datos (80% entrenamiento, 20% prueba)
X_train_raw, X_test_raw, y_train_og, y_test_og = train_test_split(
    X_todo, y_todo_og, test_size=0.20, seed=42)

#Estandarizacion Z-Score (usando solo X_train para evitar data leakage)
mean = X_train_raw.mean(axis=0)
std = X_train_raw.std(axis=0)
std = np.where(std == 0, 1.0, std)

X_train = (X_train_raw - mean) / std
X_test = (X_test_raw - mean) / std

#Binarizacion One-vs-Rest (OvR) para el entrenamiento
y_train_c1 = np.where(y_train_og == 1, 1, 0)
y_train_c2 = np.where(y_train_og == 2, 1, 0)
y_train_c3 = np.where(y_train_og == 3, 1, 0)

print(f"Dimensiones de X_train estandarizado: {X_train.shape}")
print(f"Distribución Clase 1 vs Resto: {np.bincount(y_train_c1)}")
print(f"Distribución Clase 2 vs Resto: {np.bincount(y_train_c2)}")
print(f"Distribución Clase 3 vs Resto: {np.bincount(y_train_c3)}")

Dimensiones de X_train estandarizado: (1700, 21)
Distribución Clase 1 vs Resto: [ 380 1320]
Distribución Clase 2 vs Resto: [1470  230]
Distribución Clase 3 vs Resto: [1550  150]


#### **2. Entrenamiento del Ensamble One-vs-Rest**
Instanciamos y entrenamos tres modelos de Regresión Logística de manera independiente. Cada modelo se especializará en detectar su propia clase en contra de todas las demás.


In [8]:
#3 modelos con los mismos hiperparametros
model_c1 = LogisticRegression(learning_rate=0.1, n_iterations=2000)
model_c2 = LogisticRegression(learning_rate=0.1, n_iterations=2000)
model_c3 = LogisticRegression(learning_rate=0.1, n_iterations=2000)

#Entrenamiento de Modelo 1 (Especialista en detectar Normales)
model_c1.fit(X_train, y_train_c1)

#Entrenamiento de Modelo 2 (Especialista en detectar Sospechosos)
model_c2.fit(X_train, y_train_c2)

#Entrenamiento de Modelo 3 (Especialista en detectar Patológicos)
model_c3.fit(X_train, y_train_c3)


#### **3. Evaluación y Decisión Multiclase (Argmax)**
Para predecir el diagnóstico de un paciente nuevo en el conjunto de prueba, le consultamos a los 3 modelos la probabilidad de que pertenezca a su clase especialista. El diagnóstico final se asigna a la clase que arroje **la probabilidad más alta** (`np.argmax`).


In [10]:
#Se obtienen las 3 probabilidades predichas por la funcion sigmoide para cada paciente
probs_c1 = model_c1.predict_proba(X_test)
probs_c2 = model_c2.predict_proba(X_test)
probs_c3 = model_c3.predict_proba(X_test)

#Agrupacion de probabilidades en una matriz de (n_samples, 3)
probs_matrix = np.column_stack((probs_c1, probs_c2, probs_c3))

#Decision Multiclase: np.argmax devuelve el indice con mayor probabilidad (0, 1 o 2)
#Se le suma 1 para regresar a la escala clínica original (1, 2 o 3)
prediction_prueba = np.argmax(probs_matrix, axis=1) + 1

#Calculo de metricas
n_classes = 3
#Ajuste a indice base cero (0,1,2) para la función custom confusion_matrix
matriz_test = confusion_matrix(y_test_og - 1, prediction_prueba - 1, n_classes)
acc_test = accuracy(y_test_og, prediction_prueba)
precisions_test = precision(matriz_test)
recalls_test = recall(matriz_test)
f1s_test = f1_score(matriz_test)

#Impresion de Evaluación Final en %
print("\n--- Resultados de Rendimiento en Prueba (OvR) ---")
print(f"Exactitud Global (Accuracy): {acc_test * 100:.2f}%\n")

print("--- Métricas detalladas por Clase ---")
nombres_clases = ["1 (Normal)", "2 (Sospechoso)", "3 (Patológico)"]
for i, nombre in enumerate(nombres_clases):
    print(f"Clase {nombre:14s} -> Precision: {precisions_test[i]*100:.2f}% | Recall: {recalls_test[i]*100:.2f}% | F1-Score: {f1s_test[i]*100:.2f}%")

print(f"\nPromedio Macro F1-Score: {np.mean(f1s_test)*100:.2f}%")
print("\nMatriz de Confusión (Filas: Real [1,2,3] | Columnas: Predicho [1,2,3]):")
print(matriz_test)


--- Resultados de Rendimiento en Prueba (OvR) ---
Exactitud Global (Accuracy): 88.97%

--- Métricas detalladas por Clase ---
Clase 1 (Normal)     -> Precision: 93.29% | Recall: 95.52% | F1-Score: 94.40%
Clase 2 (Sospechoso) -> Precision: 66.13% | Recall: 63.08% | F1-Score: 64.57%
Clase 3 (Patológico) -> Precision: 85.71% | Recall: 69.23% | F1-Score: 76.60%

Promedio Macro F1-Score: 78.52%

Matriz de Confusión (Filas: Real [1,2,3] | Columnas: Predicho [1,2,3]):
[[320  14   1]
 [ 22  41   2]
 [  1   7  18]]
